Agent、AgentExecutor的创建

|                | 环节1：创建Agent                          | 环节2：创建AgentExecutor          |
|----------------|-------------------------------------------|-----------------------------------|
| 方式1：传统方式 | 使用 AgentType 指定                       | initialize_agent()                |
| 方式2：通用方式 | create_xxx_agent()<br>比如: create_react_agent()、create_tool_calling_agent() | 调用AgentExecutor()<br>构造方法   |

# 1 Agent的类型

Agents的核心类型有两种模式：

方式1：Funcation Call模式

方式2：ReAct 模式

## 1.1 FUNCATION_CALL模式

基于结构化函数调用（如 OpenAI Function Calling）

直接生成工具调用参数（ JSON 格式）

效率更高，适合工具明确的场景

典型 AgentType：

In [ ]:
from langchain_classic.agents import AgentType
#第1种：
AgentType.OPENAI_FUNCTIONS

#第2种：
AgentType.OPENAI_MULTI_FUNCTIONS

工作流程示例：

In [ ]:
# 第1步：找到Search工具：{"tool": "Search", "args": {"query": "LangChain最新版本"}}

# 第2步：执行Search工具

# ======================================

# 第1步：找打scrape_website工具：{"tool": "Search", "args": {"target": "LangChain最新版本","url":"要
# 抓取的网站地址"}}

# 第2步：执行scrape_website工具

## 1.2 ReAct模式

基于 文本推理 的链式思考（Reasoning + Acting），具备反思和自我纠错能力。

推理（Reasoning）：分析当前状态，决定下一步行动
行动（Acting）：调用工具并返回结果

通过自然语言描述决策过程

适合需要明确推理步骤的场景。例如智能客服、问答系统、任务执行等。

典型 AgentType：

In [ ]:
from langchain_classic.agents import  AgentType
#第1种：零样本推理(可以在没有预先训练的情况下尝试解决新的问题)
AgentType.ZERO_SHOT_REACT_DESCRIPTION

#第2种：无记忆对话
AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION

#第3种：带记忆对话
AgentType.CONVERSATIONAL_REACT_DESCRIPTION

工作流程示例：

问题：我想要查询xxx

思考：我需要先搜索最新信息 → 行动：调用Search工具 → 观察：获得3个结果 →

思考：需要抓取第一个链接 → 行动：调用scrape_website工具...→ 观察：获得工具结果

最后：获取结果

| 特性       | Function Call模式                     | ReAct模式                     |
|------------|---------------------------------------|-------------------------------|
| 底层机制   | 结构化函数调用                        | 自然语言推理                  |
| 输出格式   | JSON/结构化数据                       | 自由文本                      |
| 适合场景   | 需要高效工具调用                      | 需要解释决策过程              |
| 典型延迟   | 较低（直接参数化调用）                | 较高（需生成完整文本）        |
| LLM要求    | 需支持函数调用（如gpt-4）             | 通用模型即可                  |

## 1.3 AgentExecutor创建方式

1、initialize_agent()方法

- **特点**：
  - 内置一些标准化模板（如 ZERO_SHOT_REACT_DESCRIPTION）
  - Agent的创建：使用 AgentType
- **优点**：快速上手（3行代码完成配置）
- **缺点**：定制化能力较弱（如提示词固定）
- **代码片段**：

In [ ]:
from langchain_classic.agents import initialize_agent, AgentType
#第1步：创建AgentExecutor
agent_executor = initialize_agent(
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    tools=[search_tool],
    verbose=True
)
#第2步：执行
agent_executor.invoke({"xxxx"})

2、AgentExecutor构造方法

- **特点**：
  - Agent的创建：使用 `create_xxx_agent`
- **优点**：
  - 可自定义提示词（如从远程hub获取或本地自定义）
  - 清晰分离Agent逻辑与执行逻辑
- **缺点**：
  - 需要更多代码
  - 需理解底层组件关系
- **代码片段**：

In [ ]:
prompt = hub.pull("hwchase17/react")
tools = [search_tool]

#第1步：创建Agent实例
agent = create_react_agent(
    llm=llm,
    prompt=prompt,
    tools=tools
)

#第2步：创建AgentExecutor实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools
)

#第3步：执行
agent_executor.invoke({"input":"xxxxx"})

## 1.4 小结创建方式

| 组件               | 传统方式                               | 通用方式                                   |
|--------------------|----------------------------------------|--------------------------------------------|
| Agent创建          | 通过 `AgentType` 枚举选择预设          | 通过 `create_xxx_agent` 显式构建           |
| AgentExecutor创建  | 通过 `initialize_agent()` 创建         | 通过 `AgentExecutor()` 创建                |
| 提示词             | 内置不可见                             | 可以自定义                                 |
| 工具集成           | AgentExecutor中显式传入                 | Agent/AgentExecutor中需显式传入            |

# 2 Agent中工具的使用之传统方式

## 2.1 单工具使用

需求：今天上海的天气怎么样

使用Tavily

### 2.1.1 方式1 使用ReAct模式

In [1]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
dotenv.load_dotenv()
#读取配置文件信息
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)
#获取一个搜索工具

#方式一 ：
# search_tool = StructuredTool.from_function(
#     func=search.run, 
#     name="Search", 
#     description="用于检索互联网上的信息")

#方式二 ：使用tool
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的信息"
)

#获取大语言模型

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0
    )
#获取Agent的实例
agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION
#获取AgentExecutor的实例
agent_executor = initialize_agent(
    llm=llm,
    agent=agent,
    tools=[search_tool],
    verbose=True #显示详细的日志信息
)
#通过AgentExecutor调用invoke（）方法来执行Agent
result = agent_executor.invoke("查询上海今天的天气情况")
print(result)

C:\Users\huan.zheng\AppData\Local\Temp\ipykernel_21080\1829368157.py:14: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=3)
C:\Users\huan.zheng\AppData\Local\Temp\ipykernel_21080\1829368157.py:41: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://



> Entering new AgentExecutor chain...
我需要查找上海今天的天气情况。  
Action: Search  
Action Input: '上海今天的天气情况'  
Observation: [{'title': 'Shanghai, Shanghai, China Weather Forecast', 'url': 'https://www.accuweather.com/en/cn/shanghai/106577/weather-forecast/106577', 'content': "2%  8 AM  55°\n\n2%  9 AM  57°\n\n2%  10 AM  59°\n\n2%  11 AM  60°\n\n2%  12 PM  64°\n\n1%  1 PM  67°\n\n0%  2 PM  70°\n\n0%\n\n## 10-Day Weather Forecast\n\nToday\n\n2/14\n\n73° 49°\n\nMostly sunny and very warm\n\nNight: Increasing clouds\n\n12% Sun\n\n2/15\n\n58° 40°\n\nCooler; breezy in the p.m.\n\nWindy with periods of rain\n\n5% Mon\n\n2/16\n\n45° 38°\n\nCooler; breezy in the morning\n\nMostly cloudy\n\n25% Tue\n\n2/17\n\n49° 38°\n\nSome sun, then turning cloudy\n\nMainly clear\n\n4% Wed\n\n2/18\n\n56° 40°\n\nHazy sunshine\n\nMainly clear\n\n0% Thu\n\n2/19\n\n58° 43°\n\nHazy sunshine\n\nClear\n\n0% Fri\n\n2/20\n\n63° 50°\n\nPlenty of sunshine\n\nClear\n\n0% Sat\n\n2/21\n\n62° 51°\n\nMostly sunny\n\nOvercast with rai

上述示例也可以写为：


In [5]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
dotenv.load_dotenv()
#读取配置文件信息
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)
#获取一个搜索工具

#方式一 ：
# search_tool = StructuredTool.from_function(
#     func=search.run, 
#     name="Search", 
#     description="用于检索互联网上的信息")

#方式二 ：使用tool
# search_tool = Tool(
#     name="Search",
#     func=search.run,
#     description="用于检索互联网上的信息"
# )

#获取大语言模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0
    )
#获取Agent的实例
agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION
#获取AgentExecutor的实例
agent_executor = initialize_agent(
    llm=llm,
    agent=agent,
    tools=[search],
    verbose=True #显示详细的日志信息
)
#通过AgentExecutor调用invoke（）方法来执行Agent
result = agent_executor.invoke({"input": "查询上海今天的天气情况"})
print(result)



> Entering new AgentExecutor chain...
我需要查找上海今天的天气情况。  
Action: tavily_search_results_json  
Action Input: "上海今天的天气情况"  
Observation: [{'title': '今天上海市天气_今日天气预报 - 墨迹天气', 'url': 'https://tianqi.moji.com/today/china/shanghai/shanghai', 'content': '小雨 03/02\n 周二 小雨  \n\n  12° 6°\n\n    多云 03/03\n 周三 小雨  \n\n  12° 8°\n\n    阴 03/04\n 周四 阴  \n\n  11° 6°\n\n    多云 03/05\n 周五 阴  \n\n  12° 6°\n\n    多云 03/06\n 周六 阴  \n\n  12° 5°\n\n    小雨 03/07\n 周日 小雨  \n\n  10° 1°\n\n    小雨 03/08\n\n 中等\n  :   紫外线\n 易发\n  :   感冒\n 不易中暑\n  :   中暑\n 不适宜\n  :   狗狗\n 不适宜\n  :   猫咪\n 不适宜\n  :   运动\n 保湿\n  :   化妆\n 舒适\n  :   穿衣\n 不适宜\n  :   广场舞\n 不适宜\n  :   钓鱼\n 良好\n  :   交通\n 不适宜\n  :   划船\n 较适宜\n  :   啤酒\n 适宜\n  :   夜生活\n 较好\n  :   心情\n 较适宜\n  :   放风筝\n 适宜\n  :   旅游\n 适宜\n  :   晨练\n 极适宜\n  :   晾晒\n 较适宜\n  :   洗车\n 良\n  :   空气污染扩散\n 较少开启\n  :   空调开启\n 适宜\n  :   约会\n 一般\n  :   美发\n 舒适\n  :   舒适度\n 干燥\n  :   路况\n 适宜\n  :   逛街\n 中等\n  :   防晒\n 不带伞\n  :   雨伞\n 无需防护\n  :   风寒\n\n## 今天适合穿\n\n天气偏凉了，墨迹天气建议您穿上厚些的外套或是保暖的

### 2.1.2 方式2：FUNCATION_CALL模式

In [6]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
import os
import dotenv

# 导入必要的额外依赖
from langchain_community.tools.tavily_search import TavilySearchResults

# 加载环境变量
dotenv.load_dotenv()

# 配置 API 密钥
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 1. 初始化 Tavily 搜索工具
search = TavilySearchResults(max_results=3)

# 2. 封装搜索工具（Tool 方式）
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的实时信息，比如天气、新闻等"
)

# 3. 初始化大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    verbose=True
)

# 4. 初始化 Agent（关键修复：参数顺序和格式）
# 注意：initialize_agent 是旧版方法，参数顺序为 tools, llm, agent
agent_executor = initialize_agent(
    tools=[search_tool],  # 第一个参数必须是 tools
    llm=llm,              # 第二个参数是 llm
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,         # 显示详细执行日志
    handle_parsing_errors=True,  # 增加错误处理
    max_iterations=5      # 限制最大迭代次数
)

# 5. 执行 Agent 并处理结果
try:
    result = agent_executor.invoke({
        "input": "查询上海今天的天气情况"  # 必须用 input 作为关键字
    })
    print("\n最终查询结果：")
    print(result["output"])
except Exception as e:
    print(f"执行出错：{type(e).__name__}: {e}")



> Entering new AgentExecutor chain...
执行出错：TypeError: langchain_core.language_models.chat_models.BaseChatModel.generate_prompt() got multiple values for keyword argument 'callbacks'


initialize_agent（旧版）在执行时会把 callbacks 作为关键字参数传给模型的 generate_prompt 方法，但新版 BaseChatModel 的 generate_prompt 方法中，callbacks 已经被包含在位置参数的 **kwargs 里，导致重复赋值。

修正后：

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_classic.agents import create_openai_functions_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import os
import dotenv
from langchain_community.tools.tavily_search import TavilySearchResults

# 加载环境变量
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 1. 初始化 Tavily 搜索工具
search = TavilySearchResults(max_results=3)
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的实时信息，比如天气、新闻、地区实时数据等"
)
tools = [search_tool]

# 2. 初始化大语言模型（新版无需额外配置 callbacks）
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. 构建新版 Agent 的 Prompt（必须包含必要的占位符）
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个智能助手，需要时使用工具获取准确信息后再回答用户问题。"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # 必须：存储思考过程
])

# 4. 创建新版 OpenAI Functions Agent（核心修复：放弃旧版 initialize_agent）
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True  # 处理解析错误
)

# 5. 执行 Agent 并输出结果
try:
    result = agent_executor.invoke({"input": "查询上海今天的天气情况"})
    print("\n✅ 最终查询结果：")
    print(result["output"])
except Exception as e:
    print(f"\n❌ 执行出错：{type(e).__name__}: {e}")



> Entering new AgentExecutor chain...

Invoking: `Search` with `上海今天的天气`


[{'title': '今天上海市天气_今日天气预报', 'url': 'https://tianqi.moji.com/today/china/shanghai/shanghai', 'content': '小雨 03/02\n 周二 小雨  \n\n  12° 6°\n\n    多云 03/03\n 周三 小雨  \n\n  12° 8°\n\n    阴 03/04\n 周四 阴  \n\n  11° 6°\n\n    多云 03/05\n 周五 阴  \n\n  12° 6°\n\n    多云 03/06\n 周六 阴  \n\n  12° 5°\n\n    小雨 03/07\n 周日 小雨  \n\n  10° 1°\n\n    小雨 03/08\n\n 中等\n  :   紫外线\n 易发\n  :   感冒\n 不易中暑\n  :   中暑\n 不适宜\n  :   狗狗\n 不适宜\n  :   猫咪\n 不适宜\n  :   运动\n 保湿\n  :   化妆\n 舒适\n  :   穿衣\n 不适宜\n  :   广场舞\n 不适宜\n  :   钓鱼\n 良好\n  :   交通\n 不适宜\n  :   划船\n 较适宜\n  :   啤酒\n 适宜\n  :   夜生活\n 较好\n  :   心情\n 较适宜\n  :   放风筝\n 适宜\n  :   旅游\n 适宜\n  :   晨练\n 极适宜\n  :   晾晒\n 较适宜\n  :   洗车\n 良\n  :   空气污染扩散\n 较少开启\n  :   空调开启\n 适宜\n  :   约会\n 一般\n  :   美发\n 舒适\n  :   舒适度\n 干燥\n  :   路况\n 适宜\n  :   逛街\n 中等\n  :   防晒\n 不带伞\n  :   雨伞\n 无需防护\n  :   风寒\n\n## 今天适合穿\n\n天气偏凉了，墨迹天气建议您穿上厚些的外套或是保暖的羊毛衫，年老体弱者可以选择保暖的摇粒绒外套。 [...] 随时随地 想查就查\n\n首页 天气 下载 资讯 关于墨迹\n\n 天气\

二者对比：ZERO_SHOT_REACT_DESCRIPTION和OPENAI_FUNCTIONS

| 对比维度         | ZERO_SHOT_REACT_DESCRIPTION                     | OPENAI_FUNCTIONS                                   |
|------------------|-------------------------------------------------|----------------------------------------------------|
| 底层机制         | 模型生成文本指令，系统解析后调用工具            | 模型直接返回JSON格式工具调用                       |
| 执行效率         | 🐢 较低（需多轮文本交互）                        | ⚡ 更高（单步完成）                                 |
| 输出可读性       | 直接显示人类可读的思考过程                      | 需查看结构化日志                                   |
| 工具参数处理     | 依赖模型文本描述准确性                          | 自动匹配工具参数结构                               |
| 兼容模型         | 所有文本生成模型                                | 仅GPT-4/Claude 3等新模型                           |
| 复杂任务表现     | 可能因文本解析失败出错                          | 更可靠（结构化保证）                               |

## 2.2 多工具使用

- **需求**：
  1. 计算特斯拉当前股价是多少？（Tavily搜索引擎）
  2. 比去年上涨了百分之几？（提示：调用PythonREPL实例的run方法）
- **核心任务**：多个（两个）工具的选择与协同使用

### 2.2.1 方式1 使用ReAct模式

注意：PythonREPL工具会执行任意 Python 代码，存在极高的安全风险（比如执行删除文件、调用恶意接口等操作），因此将其从稳定的 langchain_community 中移除，转移到了「实验性」的 langchain_experimental 模块中，以此提醒开发者注意其安全风险。
首先需要安装 langchain_experimental 包（这是独立于核心包的）：
pip install langchain-experimental

In [8]:
# 1.导入相关依赖
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.utilities.python import PythonREPL

dotenv.load_dotenv()

# 2. 设置 TAVILY_API 密钥
#读取配置文件信息
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 3.定义搜索工具
# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)
#获取一个搜索工具

#方式一 ：
# search_tool = StructuredTool.from_function(
#     func=search.run, 
#     name="Search", 
#     description="用于检索互联网上的信息")

#方式二 ：使用tool
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的信息"
)

# 4.定义计算工具
python_repl = PythonREPL() # LangChain封装的工具类可以进行数学计算

calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如计算百分比变化"
)


#获取大语言模型

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0
    )
#获取Agent的实例
agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION
#获取AgentExecutor的实例
agent_executor = initialize_agent(
    tools=[search_tool, calc_tool],
    llm=llm,
    agent=agent,
    verbose=True #显示详细的日志信息
)


#通过AgentExecutor调用invoke（）方法来执行Agent
query = "特斯拉当前股价是多少？比去年上涨了百分之几？"
result=agent_executor.invoke(query)
print(f"查询结果: {result}")

C:\Users\huan.zheng\AppData\Local\Temp\ipykernel_23868\1188743949.py:19: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=3)
C:\Users\huan.zheng\AppData\Local\Temp\ipykernel_23868\1188743949.py:56: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://



> Entering new AgentExecutor chain...
我需要先查找特斯拉当前的股价，然后再查找去年的股价，以计算上涨的百分比。  
Action: Search  
Action Input: "特斯拉当前股价"  
Observation: [{'title': 'Tesla: TSLA Stock Price Quote & News | Robinhood', 'url': 'https://robinhood.com/us/en/stocks/TSLA/', 'content': "Founded2003\n\nEmployees\n134,785\n\nEmployees134,785\n\n## TSLA Key Statistics\n\nMarket cap\n1.51T\n\nMarket cap1.51T\n\nPrice-Earnings ratio\n382.84\n\nPrice-Earnings ratio382.84\n\nDividend yield\n—\n\nDividend yield—\n\nAverage volume\n58.18M\n\nAverage volume58.18M\n\nHigh today\n$414.70\n\nHigh today$414.70\n\nLow today\n$405.50\n\nLow today$405.50\n\nOpen price\n$408.30\n\nOpen price$408.30\n\nVolume\n31.95M\n\nVolume31.95M\n\n52 Week high\n$498.83\n\n52 Week high$498.83\n\n52 Week low\n$214.25\n\n52 Week low$214.25\n\n## Stock Snapshot\n\nThe current Tesla(TSLA) stock price is $407.87, with a market capitalization of 1.51T. The stock trades at a price-to-earnings (P/E) ratio of 382.84. [...] During the trading session on

Python REPL can execute arbitrary code. Use with caution.


我找到了特斯拉去年的股价，最低为214.25美元，最高为498.83美元。为了计算特斯拉当前股价407.87美元相较于去年的涨幅，我需要使用去年的最低价214.25美元进行计算。  
Action: Calculator  
Action Input: "((407.87 - 214.25) / 214.25) * 100"  
Observation: 
Thought:我已经计算出特斯拉当前股价相较于去年的最低价的涨幅。  
Final Answer: 特斯拉当前股价为407.87美元，相较于去年的最低价214.25美元上涨了约90.0%。

> Finished chain.
查询结果: {'input': '特斯拉当前股价是多少？比去年上涨了百分之几？', 'output': '特斯拉当前股价为407.87美元，相较于去年的最低价214.25美元上涨了约90.0%。'}


### 2.2.2 方式2：FUNCATION_CALL模式

AgentType是 FUNCATION_CALL

In [ ]:
# 1.导入相关依赖
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.utilities.python import PythonREPL

dotenv.load_dotenv()

# 2. 设置 TAVILY_API 密钥
#读取配置文件信息
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 3.定义搜索工具
# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)
#获取一个搜索工具

#方式一 ：
# search_tool = StructuredTool.from_function(
#     func=search.run, 
#     name="Search", 
#     description="用于检索互联网上的信息")

#方式二 ：使用tool
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的信息"
)

# 4.定义计算工具
python_repl = PythonREPL() # LangChain封装的工具类可以进行数学计算

calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如计算百分比变化"
)


#获取大语言模型

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0
    )
#获取Agent的实例
agent = AgentType.OPENAI_FUNCTIONS  #唯一修改的位置
#获取AgentExecutor的实例
agent_executor = initialize_agent(
    tools=[search_tool, calc_tool],
    llm=llm,
    agent=agent,
    verbose=True #显示详细的日志信息
)


#通过AgentExecutor调用invoke（）方法来执行Agent
# query = "特斯拉当前股价是多少？比去年上涨了百分之几？"
result=agent_executor.invoke({"input":"特斯拉当前股价是多少？比去年上涨了百分之几？"})
print(f"查询结果: {result}")



> Entering new AgentExecutor chain...


TypeError: langchain_core.language_models.chat_models.BaseChatModel.generate_prompt() got multiple values for keyword argument 'callbacks'

修复后：

In [ ]:
# 1.导入相关依赖
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
import os
import dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.utilities.python import PythonREPL
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_openai_functions_agent, AgentExecutor

dotenv.load_dotenv()

# 2. 设置 TAVILY_API 密钥
#读取配置文件信息
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 3.定义搜索工具
# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)
#获取一个搜索工具

#方式一 ：
# search_tool = StructuredTool.from_function(
#     func=search.run, 
#     name="Search", 
#     description="用于检索互联网上的信息")

#方式二 ：使用tool
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的信息"
)

# 4.定义计算工具
python_repl = PythonREPL() # LangChain封装的工具类可以进行数学计算

calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如计算百分比变化"
)

tools = [search_tool, calc_tool]
#获取大语言模型

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0
    )
#获取Agent的实例
agent = AgentType.OPENAI_FUNCTIONS  #唯一修改的位置
#获取AgentExecutor的实例
agent_executor = initialize_agent(
    tools=[search_tool, calc_tool],
    llm=llm,
    agent=agent,
    verbose=True #显示详细的日志信息
)


# 3. 构建新版 Agent 的 Prompt（必须包含必要的占位符）
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个智能助手，需要时使用工具获取准确信息后再回答用户问题。"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # 必须：存储思考过程
])

# 4. 创建新版 OpenAI Functions Agent（核心修复：放弃旧版 initialize_agent）
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True  # 处理解析错误
)

# 5. 执行 Agent 并输出结果
try:
    result = agent_executor.invoke({"input": "特斯拉当前股价是多少？比去年上涨了百分之几？"})
    print("\n✅ 最终查询结果：")
    print(result["output"])
except Exception as e:
    print(f"\n❌ 执行出错：{type(e).__name__}: {e}")




> Entering new AgentExecutor chain...

Invoking: `Search` with `特斯拉当前股价`


[{'title': 'Tesla: TSLA Stock Price Quote & News | Robinhood', 'url': 'https://robinhood.com/us/en/stocks/TSLA/', 'content': "Founded2003\n\nEmployees\n134,785\n\nEmployees134,785\n\n## TSLA Key Statistics\n\nMarket cap\n1.51T\n\nMarket cap1.51T\n\nPrice-Earnings ratio\n382.84\n\nPrice-Earnings ratio382.84\n\nDividend yield\n—\n\nDividend yield—\n\nAverage volume\n58.18M\n\nAverage volume58.18M\n\nHigh today\n$414.70\n\nHigh today$414.70\n\nLow today\n$405.50\n\nLow today$405.50\n\nOpen price\n$408.30\n\nOpen price$408.30\n\nVolume\n31.95M\n\nVolume31.95M\n\n52 Week high\n$498.83\n\n52 Week high$498.83\n\n52 Week low\n$214.25\n\n52 Week low$214.25\n\n## Stock Snapshot\n\nThe current Tesla(TSLA) stock price is $407.87, with a market capitalization of 1.51T. The stock trades at a price-to-earnings (P/E) ratio of 382.84. [...] During the trading session on 2026-02-20, Tesla(TSLA) shares reached a daily high of $4

## 2.3 自定义函数与工具（拓展）

需求：计算3的平方，Agent自动调用工具完成

In [20]:
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_openai import ChatOpenAI

# 1. 定义工具 - 计算器（要求字符串输入）
def simple_calculator(expression: str) -> str:
    """基础数学计算工具，支持加减乘除和幂运算参数:expression: 数学表达式字符串，如 "3+5" 或 "2**3"返回:计算结果字符串或错误信息"""
    print(f"\n[工具调用] 计算表达式: {expression}")
    print("只因为在人群中多看了你一眼，确认下你调用了我^_^")
    return str(eval(expression))

# 2. 创建工具对象
math_calculator_tool = Tool(
    name="Math_Calculator", # 工具名称（Agent将根据名称选择工具）
    func=simple_calculator, # 工具调用的函数
    description="用于数学计算，输入必须是纯数学表达式（如'3+5'或'3**2'表示平方）。不支持字母或特殊符号" # 关键：明确输入格式要求
)

# 3. 初始化大模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 4. 初始化AgentExecutor（使用零样本React模式、增加超时设置）
agent_executor = initialize_agent(
    tools=[math_calculator_tool], # 可用的工具列表
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, # 简单指令模式
    verbose=True # 关键参数！在控制台显示详细的推理过程
)

# 5. 测试工具调用（添加异常捕获）
print("=== 测试：正常工具调用 ===")
response = agent_executor.invoke("计算3的平方") # 向Agent提问
print("最终答案:", response)

=== 测试：正常工具调用 ===


> Entering new AgentExecutor chain...
我需要计算3的平方，这可以用数学表达式表示为'3**2'。  
Action: Math_Calculator  
Action Input: '3**2'  
[工具调用] 计算表达式: '3**2'
只因为在人群中多看了你一眼，确认下你调用了我^_^

Observation: 3**2
Thought:我需要计算3的平方，结果是9。  
Final Answer: 9

> Finished chain.
最终答案: {'input': '计算3的平方', 'output': '9'}


# 3 Agent中工具的使用之通用方式

## 3.1 使用FUNCTION_CALL模式

In [27]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# 2.定义搜索化工具
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

search = TavilySearchResults(max_results=3) #这里可以直接将TavilySearchResults实例作为工具传入AgentExecutor，AgentExecutor会自动识别并调用它的run方法

# 3.自定义提示词模版
prompt = ChatPromptTemplate.from_messages([
    ("system","您是一位乐于助人的助手，请务必使用tavily_search_results_json 工具来获取信息。"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 4.定义LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#5.创建Agent对象
agent = create_tool_calling_agent(
    llm = llm,
    tools = [search],
    prompt = prompt
)

# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(agent=agent, tools=[search], verbose=True)
# 7.执行Agent
result = agent_executor.invoke({"input": "查询上海今天的天气情况"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '上海今天的天气情况'}`


[{'title': '今天上海市天气_今日天气预报 - 墨迹天气', 'url': 'https://tianqi.moji.com/today/china/shanghai/shanghai', 'content': '多云 03/03\n 周三 小雨  \n\n  13° 2°\n\n    多云 03/04\n 周四 晴  \n\n  11° 4°\n\n    晴 03/05\n 周五 晴  \n\n  13° 8°\n\n    阴 03/06\n 周六 小雨  \n\n  14° 7°\n\n    小雨 03/07\n 周日 阴  \n\n  11° 4°\n\n    阴 03/08\n 周一 阴  \n\n  11° 4°\n\n    阴 03/09\n\n 弱\n  :   紫外线\n 极易发\n  :   感冒\n 不易中暑\n  :   中暑\n 适宜\n  :   狗狗\n 不适宜\n  :   猫咪\n 不适宜\n  :   运动\n 控油\n  :   化妆\n 温凉\n  :   穿衣\n 较适宜\n  :   广场舞\n 较适宜\n  :   钓鱼\n 较好\n  :   交通\n 适宜\n  :   划船\n 较不适宜\n  :   啤酒\n 适宜\n  :   夜生活\n 较好\n  :   心情\n 较适宜\n  :   放风筝\n 较适宜\n  :   旅游\n 适宜\n  :   晨练\n 基本适宜\n  :   晾晒\n 不宜\n  :   洗车\n 良\n  :   空气污染扩散\n 开启制暖\n  :   空调开启\n 适宜\n  :   约会\n 一般\n  :   美发\n 舒适\n  :   舒适度\n 干燥\n  :   路况\n 较适宜\n  :   逛街\n 较弱\n  :   防晒\n 不带伞\n  :   雨伞\n 凉\n  :   风寒\n\n## 今天适合穿\n\n潮湿闷热，墨迹天气建议您穿着吸湿排汗的夏装，棉麻质地的短打扮是不错的选择。 [...] 随时随地 想查就查\n

In [ ]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# 2.定义搜索化工具
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

search = TavilySearchResults(max_results=3) 
# 方式二：使用Tool封装TavilySearchResults实例
search_tool = Tool(
    name="Search",
    func=search.run, # 直接传入TavilySearchResults实例的run方法
    description="用于检索互联网上的实时信息，比如天气、新闻等"
)

# 3.自定义提示词模版
prompt = ChatPromptTemplate.from_messages([
    ("system","您是一位乐于助人的助手，请务必使用 Search 工具来获取信息。"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 4.定义LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#5.创建Agent对象
agent = create_tool_calling_agent(
    llm = llm,
    tools = [search],
    prompt = prompt
)

# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(agent=agent, tools=[search], verbose=True)
# 7.执行Agent
result = agent_executor.invoke({"input": "查询上海今天的天气情况"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '上海今天的天气'}`


[{'title': '【上海市明天天气】_明天上海市天气_明日天气预报- 墨迹天气', 'url': 'https://tianqi.moji.com/tommorrow/china/shanghai/shanghai', 'content': '随时随地 想查就查\n\n首页 天气 下载 资讯 关于墨迹\n\n 天气\n 中国\n 上海市\n 上海市\n\n上海市， 上海市， 中国\n\n8 12\n\n小雨~~转~~阴 09:02更新\n\n2026年02月24日 丙午[马]年 正月初八\n\n热门时景 更多\n\n ## 四川省凉山彝族自治州盐源县泸沽湖镇亚泸路泸沽湖风景名胜区\n ## 安徽省黄山市休宁县溪口镇詹家山\n\n附近地区 更多\n\n 嘉定区\n 浦东新区\n 金山区\n 松江区\n 青浦区\n 南汇区（现浦东新区）\n 奉贤区\n 崇明区\n\n附近景点 更多\n\n 枫泾古镇\n 东方明珠塔\n 陆家嘴中心绿地\n 上海自然博物馆(上海科技馆分馆)\n 上海科技馆\n 上海野生动物园\n 兰心戏院旧址\n 上海城市规划展示馆\n\n## 15天预报\n\n10:02更新\n\n 周日 晴  \n\n  20° 7°\n\n    晴 02/22\n 周一 多云  \n\n  15° 9°\n\n    小雨 02/23\n 周二 小雨  \n\n  12° 8°\n\n    阴 02/24\n 周三 阴  \n\n  13° 8°\n\n    阴 02/25\n 周四 小雨  \n\n  14° 9°\n\n    中雨 02/26\n 周五 小雨  \n\n  13° 7°\n\n    多云 02/27\n 周六 阴  \n\n  15° 9°\n\n    小雨 02/28\n 周日 小雨  \n\n  14° 8°\n\n    中雨 03/01\n 周一 小雨  \n\n  15° 9°\n\n    小雨 03/02\n 周二 阴  \n\n  14° 8°\n\n    多云 03/03\n 周三 小雨  

注意：agent_scratchpad必须声明，它用于存储和传递Agent的思考过程。比如，在调用链式工具时
（如先搜索天气再推荐行程）， agent_scratchpad 保留所有历史步骤，避免上下文丢失。format方法
会将intermediate_steps转换为特定格式的字符串，并赋值给agent_scratchpad变量。如果不传递
intermediate_steps参数，会导致KeyError: 'intermediate_steps'错误。

## 3.2 使用ReAct模式

### 体会1：使用PromptTemplate


提示词要体现可以使用的工具、用户输入和agent_scratchpad。

远程的提示词模版通过https://smith.langchain.com/hub/hwchase17获取

举例：https://smith.langchain.com/hub/hwchase17/react，这个模板是专为ReAct模式设计的
提示模板。这个模板中已经有聊天对话键tools 、tool_names 、agent_scratchpad

In [ ]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
# 2.定义搜索化工具
tools = [TavilySearchResults(max_results=1,tavily_api_key=os.getenv("TAVILY_API_KEY"))]
# 3.自定义提示词模版
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""
prompt = PromptTemplate.from_template(template)

# 4.定义LLM
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 5.创建Agent对象
agent = create_react_agent(llm, tools, prompt)
# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True
    )  

# 7.执行Agent  
agent_executor.invoke({"input": "上海今天的天气情况如何？"})



> Entering new AgentExecutor chain...
Action: tavily_search_results_json  
Action Input: 上海今天的天气情况  
[{'title': '【上海天气预报】上海天气预报7天,10天,15天_全国天气网', 'url': 'https://tianqi.so.com/weather/101020100', 'content': '# 全国天气网\n\n 首页\n 国内天气\n 空气质量\n 国际天气\n 景点天气\n 天气新闻\n 专业天气\n\n收藏\n\n上海\n\n[切换]\n\n当前时间：2026-02-24周二02:27\n\n空气 优东北风 2级\n\n 今天 (02-24)\n\n  北风 微风\n\n 明天 (02-25)\n\n  东北风 微风\n\n 周四 (02-26)\n\n  东南风 微风\n\n 周五 (02-27)\n\n  东北风 微风\n\n 周六 (02-28)\n\n  9/15℃\n\n  东风 微风\n\n 周日 (03-01)\n\n  小雨转中雨\n\n  10/14℃\n\n  东风 微风\n\n 周一 (03-02)\n\n  阵雨\n\n  8/15℃\n\n  西北风 微风\n\n 周二 (03-03)\n\n  阵雨转多云\n\n  6/14℃\n\n  东风 微风\n\n 周三 (03-04)\n\n  阵雨转多云\n\n  2/13℃\n\n  北风 微风\n\n 周四 (03-05)\n\n  阴\n\n  4/9℃\n\n  北风 3-5级\n\n 周五 (03-06)\n\n  阴\n\n  7/10℃\n\n  东南风 3-5级\n\n 周六 (03-07)\n\n  阴转阵雨\n\n  6/14℃\n\n  东北风 微风\n\n 周日 (03-08)\n\n  阴转多云\n\n  3/11℃\n\n  东北风 微风\n\n 周一 (03-09)\n\n  阴转阵雨\n\n  5/13℃\n\n  东北风 微风\n\n45日天气\n\n15日天气\n\n今日天气\n\n明日天气\n\n### 当前天气信息\n\n天气：中雨 9℃\n\n体感：4℃\n\n风向：东北风\n\n风力：2级\n\n气压：1017\n\n

{'input': '上海今天的天气情况如何？', 'output': '上海今天的天气情况是中雨，气温9°C，体感温度4°C，湿度93%，空气质量良好。'}

In [17]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
# 2.定义搜索化工具
tools = [TavilySearchResults(max_results=1,tavily_api_key=os.getenv("TAVILY_API_KEY"))]
# 3.自定义提示词模版
template = """请尽你所能回答以下问题。你可以使用以下工具：
{tools}
请使用以下格式：
Question:你必须回答的输入问题
Thought:你需要始终思考接下来要做什么
Action:需要执行的操作，必须是 {tool_names}中的一个
Action Input:执行该操作的输入内容
Observation:操作得到的结果
……（上述 思考 / 动作 / 动作输入 / 观察 步骤可重复 N 次）
Thought:我现在已经知道最终答案了
Final Answer:原始输入问题的最终答案
开始！
Question:{question}
Thought:{agent_scratchpad}
"""
prompt = PromptTemplate.from_template(template)

# 4.定义LLM
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 5.创建Agent对象
agent = create_react_agent(llm, tools, prompt)
# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True
    )  

# 7.执行Agent  
agent_executor.invoke({"question": "上海今天的天气情况如何？"})



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: 上海今天的天气情况
[{'title': '2月25日上海天气晴到多云2°C-10°C', 'url': 'https://m.sh.bendibao.com/news/268009.html', 'content': '# 2月25日上海天气晴到多云2°C-10°C\n\n导语 今天（2月25日）晴到多云。早晨气温有点低，城郊差异较大，预计今天最高气温10℃，注意防风保暖，多多补充水份!\n\n今天早晨气温有点低，城郊差异较大，像金山、奉贤最低仅1、2℃上下，其余地区多在4~6℃，白天天气晴好，风力较大，偏北风4~5级阵风6级，预计今天最高气温10℃，相对湿度80%~30%，注意防风保暖，多多补充水份!\n\n天气趋势：\n\n\u3000\u3000周日天气与今天类似，晴到多云，风力减小，偏北风3~4级，气温在2~10℃，郊区早晨气温会更低，可能又会跌至冰点。下周天气总体来讲是晴多雨少，就周二前后有一次小雨过程，白天气温逐渐回升，预计下周最高气温在16~17℃，春天的脚步近了~\n\n\u3000\u3000注：本文图文信息源于上海天气网。\n\n分享本文到：\n\n关闭\n\n内容来源上海本地宝\n\n上海本地宝\n\n+ 关注\n\n分享 返回首页\n\n   \n反馈 提问', 'score': 0.99589807}]Action: tavily_search_results_json
Action Input: 上海今天的天气情况
[{'title': '2月25日上海天气晴到多云2°C-10°C', 'url': 'https://m.sh.bendibao.com/news/268009.html', 'content': '# 2月25日上海天气晴到多云2°C-10°C\n\n导语 今天（2月25日）晴到多云。早晨气温有点低，城郊差异较大，预计今天最高气温10℃，注意防风保暖，多多补充水份!\n\n今天早晨气温有点低，城郊差异较大，像金山、奉贤最低仅1、2℃上下，其余地区多在4~6℃，白天天气晴好，风力较大，偏北风4~5级阵风6级，预计今天最高气温10℃，相对湿度80%~

{'question': '上海今天的天气情况如何？',
 'output': '上海今天的天气情况是晴到多云，早晨气温较低，预计最高气温为10℃，风力较大，偏北风4~5级，阵风可达6级。'}

需要注意：Question:/Thought:/Action:/Action Input:/Observation:
最好都要英文，不然会识别报错。

上述示例可以按如下修改：远程调用

In [23]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langchain_classic import hub
import os
import dotenv
from langchain_openai import ChatOpenAI
# 2.定义搜索化工具
tools = [TavilySearchResults(max_results=1,tavily_api_key=os.getenv("TAVILY_API_KEY"))]

# 3.使用Langchain Hub中的预设ReAct模版
prompt_template = hub.pull("hwchase17/react")

# 4.定义LLM
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 5.创建Agent对象
agent = create_react_agent(llm, tools, prompt)
# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True
    )  

# 7.执行Agent  
agent_executor.invoke({"question": "上海今天的天气情况如何？"})



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: 上海今天的天气情况
[{'title': '上海, 上海市, 中國三日天氣預報 - AccuWeather', 'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577', 'content': '返回 \n\n# 上海, 上海市\n\n54°F   \n\n54°\n\n建立您的帳戶  解鎖延長的每日與小時預報——全部用你的免費帳號。\n\n走吧\n\n已有帳戶？  登入\n\n上海, 上海市 天氣\n\n今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動\n\n全球範圍\n\n### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊\n\n## 今天  每小時   每天   雷達   MinuteCast®   每月   空氣品質   健康與活動\n\n## 目前天氣\n\nAM2:23\n\n54°F\n\nRealFeel® 54°\n\n多雲  更多詳情\n\n風 東南偏南 4英里/小时\n\n風速 4英里/小时\n\n空氣品質 不健康 ## 展望未來\n\n降雨 週日晚上\n\n## 上海 氣象雷達圖\n\nStatic Radar Temporarily Unavailable\n\nThank you for your patience as we work to get everything up and running again.\n\nRefresh Page\n\n有雲 溫度\n\n## 每小時預測\n\n上午3时  52°\n\n7%  上午4时  52°\n\n7%  上午5时  53°\n\n7%  上午6时  54°\n\n6%  上午7时  54°\n\n2%  上午8时  55°\n\n2%  上午9时  57°\n\n2%  上午10时  59°\n\n2%  上午11时  60°\n\n2%  下午12时  64°\n\n1%  下午1时  67°\n\n0

{'question': '上海今天的天气情况如何？',
 'output': '上海今天的天气情况是多云，当前温度为54°F（约12°C），空气质量不健康。预计今晚有降雨。'}

### 体会2：使用ChatPromptTemplate

In [31]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
# 2.定义搜索化工具
tools = [TavilySearchResults(max_results=1,tavily_api_key=os.getenv("TAVILY_API_KEY"))]
# 3.自定义提示词模版
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!
"""
prompt = ChatPromptTemplate.from_messages([
    ("system",template),
    ("human", "{input}"),
    ("system", "{agent_scratchpad}"),#这里不能使用placeholder占位符了，因为create_react_agent内部已经默认将agent_scratchpad作为思考过程的占位符了，如果再使用placeholder占位符会导致agent_scratchpad被覆盖，无法正确记录思考过程
]
)

# 4.定义LLM
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 5.创建Agent对象
agent = create_react_agent(llm, tools, prompt)
# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True
    )  

# 7.执行Agent  
result = agent_executor.invoke({"input":"上海今天的天气情况如何？"})
print(f"查询结果: {result}")



> Entering new AgentExecutor chain...
Question: 上海今天的天气情况如何？
Thought: 我需要查找上海今天的天气情况。
Action: tavily_search_results_json
Action Input: 上海今天的天气[{'title': '上海, 上海市, 中國三日天氣預報 - AccuWeather', 'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577', 'content': '返回 \n\n# 上海, 上海市\n\n54°F   \n\n54°\n\n建立您的帳戶  解鎖延長的每日與小時預報——全部用你的免費帳號。\n\n走吧\n\n已有帳戶？  登入\n\n上海, 上海市 天氣\n\n今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動\n\n全球範圍\n\n### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊\n\n## 今天  每小時   每天   雷達   MinuteCast®   每月   空氣品質   健康與活動\n\n## 目前天氣\n\nAM2:23\n\n54°F\n\nRealFeel® 54°\n\n多雲  更多詳情\n\n風 東南偏南 4英里/小时\n\n風速 4英里/小时\n\n空氣品質 不健康 ## 展望未來\n\n降雨 週日晚上\n\n## 上海 氣象雷達圖\n\nStatic Radar Temporarily Unavailable\n\nThank you for your patience as we work to get everything up and running again.\n\nRefresh Page\n\n有雲 溫度\n\n## 每小時預測\n\n上午3时  52°\n\n7%  上午4时  52°\n\n7%  上午5时  53°\n\n7%  上午6时  54°\n\n6%  上午7时  54°\n\n2%  上午8时  55°\n\n2%  上午9时  57°\n\n2%  上午10时  59°\n\n2%  上午11时

小结：

1、传统方式：相较于通用方式来讲，不用提供提示词模板

2、对于通用模式：

FUNCTION_CALL模式：在创建Agent时，推荐使用ChatPromptTemplate

ReAct模式：在创建Agent时，可以使用ChatPromptTemplate、PromptTemplate，推荐使用PromptTemplate

# 4 Agent嵌入记忆组件

需求：上海明天的天气怎么样？上海呢？（通过两次对话实现）

## 4.1 传统方式之ReAct模式

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic.memory import ConversationBufferMemory
dotenv.load_dotenv()
#读取配置文件信息
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)
#获取一个搜索工具
# 使用tool
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的信息"
)

#获取大语言模型

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0
    )
#获取一个记忆实例：ConversationBufferMemory（对话缓冲记忆）
memory = ConversationBufferMemory(
    memory_key="chat_history", #这里必须设置chat_history，因为提示词模板中使用的是chat_history，记忆存储的键名，AgentExecutor会将对话历史存储在这个键下
    return_messages=True
    )


#获取Agent的实例
agent = AgentType.CONVERSATIONAL_REACT_DESCRIPTION

#获取AgentExecutor的实例
agent_executor = initialize_agent(
    llm=llm,
    agent=agent,
    tools=[search_tool],
    verbose=True, #显示详细的日志信息
    memory=memory #将记忆传入AgentExecutor
)
#通过AgentExecutor调用invoke（）方法来执行Agent
result = agent_executor.invoke({"input": "查询上海今天的天气情况"})
print(result)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: Search
Action Input: 上海今天的天气情况
```
Observation: [{'title': '2月25日上海天气晴到多云2°C-10°C', 'url': 'https://m.sh.bendibao.com/news/268009.html', 'content': '# 2月25日上海天气晴到多云2°C-10°C\n\n导语 今天（2月25日）晴到多云。早晨气温有点低，城郊差异较大，预计今天最高气温10℃，注意防风保暖，多多补充水份!\n\n今天早晨气温有点低，城郊差异较大，像金山、奉贤最低仅1、2℃上下，其余地区多在4~6℃，白天天气晴好，风力较大，偏北风4~5级阵风6级，预计今天最高气温10℃，相对湿度80%~30%，注意防风保暖，多多补充水份!\n\n天气趋势：\n\n\u3000\u3000周日天气与今天类似，晴到多云，风力减小，偏北风3~4级，气温在2~10℃，郊区早晨气温会更低，可能又会跌至冰点。下周天气总体来讲是晴多雨少，就周二前后有一次小雨过程，白天气温逐渐回升，预计下周最高气温在16~17℃，春天的脚步近了~\n\n\u3000\u3000注：本文图文信息源于上海天气网。\n\n分享本文到：\n\n关闭\n\n内容来源上海本地宝\n\n上海本地宝\n\n+ 关注\n\n分享 返回首页\n\n   \n反馈 提问', 'score': 0.76554126}, {'title': '上海天气预报,上海7天天气预报,上海15天天气预报,上海天气查询', 'url': 'https://www.weather.com.cn/weather/101020100.shtml', 'content': '首页 预报 预警 雷达 云图 天气地图 专业产品 资讯 视频 节气 我的天空\n\n更多\n\n台风路径 空间天气 图片 专题 环境 旅游 碳中和 气象科普 一带一路 产创平台\n\n国内) 本地) 国际)\n\n:   北京 上海 成都 杭州 南京 天津 深圳 重庆 西安 广州 青岛 武汉\n\n:   故宫 阳朔漓江 龙门石窟 野三坡 颐和园 九寨沟

In [37]:
result = agent_executor.invoke({"input": "上海呢？"})
print(result)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? No
AI: 上海是中国的一座国际大都市，位于长江入海口，拥有丰富的历史和文化。它是中国的经济、金融、贸易和航运中心之一。上海以其现代化的天际线、繁华的商业区和丰富的美食而闻名。你想了解关于上海的哪个方面呢？例如旅游、文化、历史或其他信息？
```

> Finished chain.
{'input': '上海呢？', 'chat_history': [HumanMessage(content='查询上海今天的天气情况', additional_kwargs={}, response_metadata={}), AIMessage(content='今天（2月25日）上海的天气情况是晴到多云，早晨气温较低，预计最高气温为10℃。早晨的气温在1-6℃之间，城郊差异较大。风力较大，偏北风4-5级，阵风可达6级。相对湿度在80%到30%之间。请注意防风保暖，并多补充水分！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='上海呢？', additional_kwargs={}, response_metadata={}), AIMessage(content='上海是中国的一座国际大都市，位于长江入海口，拥有丰富的历史和文化。它是中国的经济、金融、贸易和航运中心之一。上海以其现代化的天际线、繁华的商业区和丰富的美食而闻名。你想了解关于上海的哪个方面呢？例如旅游、文化、历史或其他信息？\n```', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'output': '上海是中国的一座国际大都市，位于长江入海口，拥有丰富的历史和文化。它是中国的经济、金融、贸易和航运中心之一。上海以其现代化的天际线、繁华的商业区和丰富的美食而闻名。你想了解关于上海的哪个方面呢？例如旅游、文化、历史或其他信息？\n```'}


举例2

In [38]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic.memory import ConversationBufferMemory
dotenv.load_dotenv()
#读取配置文件信息
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)
#获取一个搜索工具
# 使用tool
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于检索互联网上的信息"
)

#获取大语言模型

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0
    )
#获取一个记忆实例：ConversationBufferMemory（对话缓冲记忆）
memory = ConversationBufferMemory(
    memory_key="chat_history", #这里必须设置chat_history，因为提示词模板中使用的是chat_history，记忆存储的键名，AgentExecutor会将对话历史存储在这个键下
    return_messages=True
    )


#获取Agent的实例
agent = AgentType.CONVERSATIONAL_REACT_DESCRIPTION

#获取AgentExecutor的实例
agent_executor = initialize_agent(
    llm=llm,
    agent=agent,
    tools=[search_tool],
    verbose=True, #显示详细的日志信息
    memory=memory #将记忆传入AgentExecutor
)
#通过AgentExecutor调用invoke（）方法来执行Agent
print("=== 查询昨天的新闻 ===")
result1 = agent_executor.invoke("昨天的新闻头条是什么？")
print(f"查询结果: {result1}")
# 第二个查询（基于上下文）
print("\n=== 用中文总结 ===")
result2 = agent_executor.invoke("用中文总结一下")
print(f"总结结果: {result2}")
# 可以继续对话
print("\n=== 继续对话 ===")
result3 = agent_executor.invoke("这些新闻对股市有什么影响？")
print(f"分析结果: {result3}")

=== 查询昨天的新闻 ===


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: Search
Action Input: 昨天的新闻头条
```
Observation: [{'title': '南加頭條| 洛杉磯| 世界新聞網', 'url': 'https://www.worldjournal.com/wj/cate/la/121360', 'content': '2026-02-17 01:20 2026-02-17 14:20\n\n### ICE催淚瓦斯鎮壓\u2004「這裡安全」華女餐館老闆收留抗議民眾 為抗議移民暨海關執法局(ICE)的執法行動，美國各地民眾走上街頭，表達自己的反對聲音。日前，明尼亞波利斯(Minneap...\n\n2026-02-17 01:18 2026-02-17 14:18\n\n### 不用學歷和經驗 這些工作年薪可達10萬 AI不易取代 媒體報導，未來就業市場機會更多在於面對面互動，難以自動化的領域。其中一些工作甚至不需要大學學曆和工作經驗，最低年薪可達6...\n\n2026-02-16 14:43 2026-02-17 03:43\n\n### 巴蘭科案迎轉折…3名海軍陸戰隊員之父\u2004驅逐令獲撤銷 洛杉磯KTLA 5報導，一名居住於加州橙縣、育有三名美國海軍陸戰隊兒子的父親巴蘭科（Narciso Barranco），...\n\n2026-02-16 01:13 2026-02-16 14:13\n\n### 國安考量…猶他州強制1中企出售逾500畝土地 為防止中國勢力染指關鍵資產，猶他州（Utah）政府日前宣布，已強制要求一家中國企業出售其位於圖埃勒縣（Tooele Co...\n\n2026-02-15 01:17 2026-02-15 14:17\n\n### 舊金山撞死1家4口 8旬華婦或免監禁 一名華裔老婦被控在舊金山街道上撞死一家四口，但法官表示，這名婦人可能不會面臨牢獄之災或社區服務。\n\n2026-02-15 01:00 2026-02-15 14:00\n\n### 防腐劑BHA有致癌風險？FDA重啟評估 Yahoo News報導，美國食品藥物管理局（FDA）日

## 4.2 通用方式

通用方式，相较于传统方式，可以提供自定义的提示词模板

### 4.2.1 使用FUNCTION_CLL模式

In [46]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# 2.定义搜索化工具
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

search = TavilySearchResults(max_results=3) 
# 方式二：使用Tool封装TavilySearchResults实例
search_tool = Tool(
    name="Search",
    func=search.run, # 直接传入TavilySearchResults实例的run方法
    description="用于检索互联网上的实时信息，比如天气、新闻等"
)

# 3.自定义提示词模版
prompt = ChatPromptTemplate.from_messages([
    ("system","您是一位乐于助人的助手，请务必使用 Search 工具来获取信息。"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
    ("system", "{chat_history}"), #存储多轮对话的历史记录 如果你没有显式传入 chat_history，Agent 会默认将其视为空列表 []
])

#提供记忆实例
memory = ConversationBufferMemory(
    memory_key="chat_history", #这里必须设置chat_history，因为提示词模板中使用chat_history
    return_messages=True)

# 4.定义LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#5.创建Agent对象
agent = create_tool_calling_agent(
    llm = llm,
    tools = [search],
    prompt = prompt
)

# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=[search], 
    verbose=True,
    memory=memory)
# 7.执行Agent
result = agent_executor.invoke({"input": "查询上海今天的天气情况"})
print(f"查询结果: {result}")



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '上海今天的天气情况'}`


[{'title': '上海, 上海市, 中國三日天氣預報 - AccuWeather', 'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577', 'content': '返回 \n\n# 上海, 上海市\n\n54°F   \n\n54°\n\n建立您的帳戶  解鎖延長的每日與小時預報——全部用你的免費帳號。\n\n走吧\n\n已有帳戶？  登入\n\n上海, 上海市 天氣\n\n今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動\n\n全球範圍\n\n### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊\n\n## 今天  每小時   每天   雷達   MinuteCast®   每月   空氣品質   健康與活動\n\n## 目前天氣\n\nAM2:23\n\n54°F\n\nRealFeel® 54°\n\n多雲  更多詳情\n\n風 東南偏南 4英里/小时\n\n風速 4英里/小时\n\n空氣品質 不健康 ## 展望未來\n\n降雨 週日晚上\n\n## 上海 氣象雷達圖\n\nStatic Radar Temporarily Unavailable\n\nThank you for your patience as we work to get everything up and running again.\n\nRefresh Page\n\n有雲 溫度\n\n## 每小時預測\n\n上午3时  52°\n\n7%  上午4时  52°\n\n7%  上午5时  53°\n\n7%  上午6时  54°\n\n6%  上午7时  54°\n\n2%  上午8时  55°\n\n2%  上午9时  57°\n\n2%  上午10时  59°\n\n2%  上午11时  60°\n\n2%  下午12时  64°\n\n1%  下

In [47]:
result = agent_executor.invoke({"input": "北京呢？"})
print(f"分析结果: {result}")



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '北京今天的天气情况'}`


[{'title': '北京天气预报,北京7天天气预报,北京15天天气预报,北京天气查询', 'url': 'https://www.weather.com.cn/weather/101010100.shtml', 'content': '首页 预报 预警 雷达 云图 天气地图 专业产品 资讯 视频 节气 我的天空\n\n更多\n\n台风路径 空间天气 图片 专题 环境 旅游 碳中和 气象科普 一带一路 产创平台\n\n国内) 本地) 国际)\n\n:   北京 上海 成都 杭州 南京 天津 深圳 重庆 西安 广州 青岛 武汉\n\n:   故宫 阳朔漓江 龙门石窟 野三坡 颐和园 九寨沟 东方明珠 凤凰古城 秦始皇陵 桃花源\n\n高球\n:   佘山 春城湖畔 华彬庄园 观澜湖 依必朗 旭宝 博鳌 玉龙雪山 番禺南沙 东方明珠\n\n<<返回 全国\n\n河北下辖区域\n\n热门城市\n\n:   曼谷 东京 首尔 吉隆坡 新加坡 巴黎 罗马 伦敦 雅典 柏林 纽约 温哥华 墨西哥城 哈瓦那 圣何塞 巴西利亚 布宜诺斯艾利斯 圣地亚哥 利马 基多 悉尼 墨尔本 惠灵顿 奥克兰 苏瓦 开罗 内罗毕 开普敦 维多利亚 拉巴特\n\n:   亚洲 欧洲 北美洲 南美洲 非洲 大洋洲\n\n全国> 北京 > 城区\n\n 今天\n 7天\n 8-15天\n 40天\n 雷达图\n\n # 24日（今天）\n\n  晴转多云\n\n  9/2℃\n\n  <3级\n # 25日（明天）\n\n  多云转小雨\n\n  10/3℃\n\n  <3级\n # 26日（后天）\n\n  雨夹雪转阴\n\n  4/0℃\n\n  <3级\n # 27日（周五）\n\n  多云转阴\n\n  6/0℃\n\n  <3级\n # 28日（周六）\n\n  阴\n\n  4/0℃\n\n  <3级\n # 1日（周日）\n\n  多云转阴\n\n  4/0℃\n\n  <3级\n # 2日（周一）\n\n  阴\n\n  5/0℃\n\n  <3级\n\n分时段预报

拓展1：如果ChatPromptTemplate.from_messages()中没有提供("placeholder", "{chat_history}")这
样操作，会怎样呢？

In [52]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# 2.定义搜索化工具
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

search = TavilySearchResults(max_results=3) 
# 方式二：使用Tool封装TavilySearchResults实例
search_tool = Tool(
    name="Search",
    func=search.run, # 直接传入TavilySearchResults实例的run方法
    description="用于检索互联网上的实时信息，比如天气、新闻等"
)

# 3.自定义提示词模版
prompt = ChatPromptTemplate.from_messages([
    ("system","您是一位乐于助人的助手，请务必使用 Search 工具来获取信息。"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
    # ("system", "{chat_history}"), #存储多轮对话的历史记录 如果你没有显式传入 chat_history，Agent 会默认将其视为空列表 []
])

#提供记忆实例
memory = ConversationBufferMemory(
    memory_key="chat_history", #这里必须设置chat_history，因为提示词模板中使用chat_history
    return_messages=True)

# 4.定义LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#5.创建Agent对象
agent = create_tool_calling_agent(
    llm = llm,
    tools = [search],
    prompt = prompt
)

# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=[search], 
    verbose=True,
    memory=memory)
# 7.执行Agent
result1 = agent_executor.invoke({"input": "查询上海今天的天气情况"})
print(f"查询结果: {result1}")
result2 = agent_executor.invoke({"input": "北京呢？"})
print(f"分析结果: {result2}")



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '上海今天的天气情况'}`


[{'title': '上海, 上海市, 中國三日天氣預報 - AccuWeather', 'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577', 'content': '返回 \n\n# 上海, 上海市\n\n54°F   \n\n54°\n\n建立您的帳戶  解鎖延長的每日與小時預報——全部用你的免費帳號。\n\n走吧\n\n已有帳戶？  登入\n\n上海, 上海市 天氣\n\n今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動\n\n全球範圍\n\n### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊\n\n## 今天  每小時   每天   雷達   MinuteCast®   每月   空氣品質   健康與活動\n\n## 目前天氣\n\nAM2:23\n\n54°F\n\nRealFeel® 54°\n\n多雲  更多詳情\n\n風 東南偏南 4英里/小时\n\n風速 4英里/小时\n\n空氣品質 不健康 ## 展望未來\n\n降雨 週日晚上\n\n## 上海 氣象雷達圖\n\nStatic Radar Temporarily Unavailable\n\nThank you for your patience as we work to get everything up and running again.\n\nRefresh Page\n\n有雲 溫度\n\n## 每小時預測\n\n上午3时  52°\n\n7%  上午4时  52°\n\n7%  上午5时  53°\n\n7%  上午6时  54°\n\n6%  上午7时  54°\n\n2%  上午8时  55°\n\n2%  上午9时  57°\n\n2%  上午10时  59°\n\n2%  上午11时  60°\n\n2%  下午12时  64°\n\n1%  下

拓展2：如果删除AgentExecutor构造方法中的memory参数，又会如何呢？

### 4.2.2 使用ReAct模式

举例1使用网络上现成的提示词模板

In [56]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langchain_classic import hub
import os
import dotenv
from langchain_openai import ChatOpenAI
# 2.定义搜索化工具
tools = [TavilySearchResults(max_results=1,tavily_api_key=os.getenv("TAVILY_API_KEY"))]

# 3.使用Langchain Hub中的预设ReAct模版
prompt_template = hub.pull("hwchase17/react-chat")

# 4.定义LLM
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#提供记忆实例
memory = ConversationBufferMemory(
    memory_key="chat_history", #这里必须设置chat_history，因为提示词模板中使用chat_history
    return_messages=True)

# 5.创建Agent对象
agent = create_react_agent(
    llm=llm, tools=tools, prompt=prompt_template)
# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True,
    memory=memory
    )  

# 7.执行Agent  
result1 = agent_executor.invoke({"input": "上海今天的天气情况如何？"})
print(f"查询结果: {result1}")
result2 = agent_executor.invoke({"input": "北京呢？"})
print(f"分析结果: {result2}")



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: tavily_search_results_json
Action Input: 上海今天的天气情况
```[{'title': '上海, 上海市, 中國三日天氣預報 - AccuWeather', 'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577', 'content': '返回 \n\n# 上海, 上海市\n\n54°F   \n\n54°\n\n建立您的帳戶  解鎖延長的每日與小時預報——全部用你的免費帳號。\n\n走吧\n\n已有帳戶？  登入\n\n上海, 上海市 天氣\n\n今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動\n\n全球範圍\n\n### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊\n\n## 今天  每小時   每天   雷達   MinuteCast®   每月   空氣品質   健康與活動\n\n## 目前天氣\n\nAM2:23\n\n54°F\n\nRealFeel® 54°\n\n多雲  更多詳情\n\n風 東南偏南 4英里/小时\n\n風速 4英里/小时\n\n空氣品質 不健康 ## 展望未來\n\n降雨 週日晚上\n\n## 上海 氣象雷達圖\n\nStatic Radar Temporarily Unavailable\n\nThank you for your patience as we work to get everything up and running again.\n\nRefresh Page\n\n有雲 溫度\n\n## 每小時預測\n\n上午3时  52°\n\n7%  上午4时  52°\n\n7%  上午5时  53°\n\n7%  上午6时  54°\n\n6%  上午7时  54°\n\n2%  上午8时  55°\n\n2%  上午9时  57°\n\n2%  上午10时  59°\n\n2%  上午11时

举例2 自定义提示词模板

In [57]:
# 1.导入相关包
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langchain_classic import hub
import os
import dotenv
from langchain_openai import ChatOpenAI
# 2.定义搜索化工具
tools = [TavilySearchResults(max_results=1,tavily_api_key=os.getenv("TAVILY_API_KEY"))]

# 3.使用自定义提示词模版
template = """Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of topics.

Overall, Assistant is a powerful tool that can help with a wide range of tasks and provide valuable insights and information on a wide range of topics. Whether you need help with a specific question or just want to have a conversation about a particular topic, Assistant is here to assist.

TOOLS:
------

Assistant has access to the following tools:

{tools}

To use a tool, please use the following format:

```
Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
```

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

```
Thought: Do I need to use a tool? No
Final Answer: [your response here]
```

Begin!

Previous conversation history:
{chat_history}

New input: {input}
{agent_scratchpad}
"""

prompt_template = PromptTemplate.from_template(
    template=template
    )

# 4.定义LLM
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
#提供记忆实例
memory = ConversationBufferMemory(
    memory_key="chat_history", #这里必须设置chat_history，因为提示词模板中使用chat_history
    return_messages=True)

# 5.创建Agent对象
agent = create_react_agent(
    llm=llm, 
    tools=tools, 
    prompt=prompt_template
    )
# 6.创建AgentExecutor执行器
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True,
    memory=memory
    )  

# 7.执行Agent  
result1 = agent_executor.invoke({"input": "上海今天的天气情况如何？"})
print(f"查询结果: {result1}")
result2 = agent_executor.invoke({"input": "北京呢？"})
print(f"分析结果: {result2}")



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: tavily_search_results_json
Action Input: 上海今天的天气情况
```[{'title': '上海, 上海市, 中國三日天氣預報 - AccuWeather', 'url': 'https://www.accuweather.com/zh/cn/shanghai/106577/weather-forecast/106577', 'content': '返回 \n\n# 上海, 上海市\n\n54°F   \n\n54°\n\n建立您的帳戶  解鎖延長的每日與小時預報——全部用你的免費帳號。\n\n走吧\n\n已有帳戶？  登入\n\n上海, 上海市 天氣\n\n今天 WinterCast 當地{stormName}追蹤 每小時 每天 雷達 MinuteCast® 每月 空氣品質 健康與活動\n\n全球範圍\n\n### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊\n\n## 今天  每小時   每天   雷達   MinuteCast®   每月   空氣品質   健康與活動\n\n## 目前天氣\n\nAM2:23\n\n54°F\n\nRealFeel® 54°\n\n多雲  更多詳情\n\n風 東南偏南 4英里/小时\n\n風速 4英里/小时\n\n空氣品質 不健康 ## 展望未來\n\n降雨 週日晚上\n\n## 上海 氣象雷達圖\n\nStatic Radar Temporarily Unavailable\n\nThank you for your patience as we work to get everything up and running again.\n\nRefresh Page\n\n有雲 溫度\n\n## 每小時預測\n\n上午3时  52°\n\n7%  上午4时  52°\n\n7%  上午5时  53°\n\n7%  上午6时  54°\n\n6%  上午7时  54°\n\n2%  上午8时  55°\n\n2%  上午9时  57°\n\n2%  上午10时  59°\n\n2%  上午11时